<a href="https://colab.research.google.com/github/jasonthurman-sys/x3d.lab/blob/main/notebooks/01_Quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# x3d.lab · 01 Quickstart

**An AI writes a 3D scene, a standards validator checks it, and you watch it render: in this notebook, on a live dashboard, and on your phone.**

[X3D](https://www.web3d.org/x3d/what-x3d) is the ISO standard for 3D on the web. An AI model can write X3D, but models invent node names and field types. The **x3d_mcp** server gives a model the real specification as [MCP](https://modelcontextprotocol.io) tools, and checks every scene against the official schemas.

| | You will |
|---|---|
| **1** | start the x3d_mcp server and a dashboard, and choose how to reach them |
| **2** | add an API key for OpenAI, Claude or Gemini (optional: everything works without one, on a sample scene) |
| **3** | ask for a scene and watch the write, validate, repair loop |
| **4** | call the MCP tools yourself |
| **5** | connect Claude Code, Claude Desktop or Cursor to *this* Colab |
| **6** | play **Break it**: inject a defect and guess which check catches it |
| **7** | open the Copilot Validator Demo (private build) |

**Runtime:** a CPU runtime is fine for this notebook. Run the cells top to bottom; cell 1 takes about a minute.

**Which x3d_mcp?** The official project is [Web3DConsortium/x3d_mcp](https://github.com/Web3DConsortium/x3d_mcp). This notebook runs the fork [hagopjay/x3d_mcp](https://github.com/hagopjay/x3d_mcp) at a pinned commit: it is ahead of upstream (exact-fix hints, ROUTE-order, colour-range and H-Anim checks, a blob store for large scenes) and pins `mcp<2`. Upstream's `pyproject.toml` allows `mcp` 2.x, where `mcp.server.fastmcp` no longer exists, so a fresh install of upstream cannot start today.

The next notebooks: **02 Lab** (Blender on an L4 or G4 GPU, a configurable stress test, Chrome DevTools profiling) and **03 Stress & Research** (fuzzing, cross-validator and cross-renderer comparisons, regression baselines).

## 1 · Start everything and choose how to reach it

This cell installs the toolkit, starts the **x3d_mcp** server (inside this Colab, never exposed directly), and starts a small **gateway** that serves the dashboard and, if you want, the MCP endpoint. Then it tests every route and prints a card with your links.

| `TUNNEL` | Who can open the dashboard | Needs |
|---|---|---|
| `none` | you, in this browser, through Colab's own proxy (private to your Google login) | nothing |
| `cloudflared` | you on any device, including your phone (QR code), and remote MCP clients | nothing (random `trycloudflare.com` URL) |
| `ngrok` | same as cloudflared, optionally on a fixed domain | a free `NGROK_AUTHTOKEN` in Colab Secrets |

Tunnel links carry a random key, and the MCP endpoint needs a separate bearer token, so a link alone never lets someone spend your API credits. Both change every time this cell runs.

`LOG_LEVEL` and `LOG_DENSITY` set the live event panel that appears under this cell (you can change both there while things run). Everything is also written to a JSONL file.

In [ ]:
#@title 1 · Start x3d_mcp, the dashboard and the MCP endpoint { display-mode: "form" }
TUNNEL = "none"  #@param ["none", "cloudflared", "ngrok"]
SHOW_DASHBOARD = "link"  #@param ["link", "iframe"]
EXPOSE_MCP = True  #@param {type:"boolean"}
LOG_LEVEL = "INFO"  #@param ["DEBUG", "INFO", "WARN", "ERROR"]
LOG_DENSITY = "compact"  #@param ["compact", "rich", "firehose"]
LOG_TO_SCREEN = True  #@param {type:"boolean"}
MAX_MODEL_RUNS = 30  #@param {type:"integer"}
NGROK_DOMAIN = ""  #@param {type:"string"}
LAB_REPO = "jasonthurman-sys/x3d.lab"
LAB_REF = "main"

import importlib.util, os, subprocess, sys
if not os.environ.get("X3DLAB_DEV") and (importlib.util.find_spec("x3dlab") is None or os.environ.get("X3DLAB_REINSTALL")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "x3dlab[notebook,tunnels] @ git+https://github.com/%s@%s" % (LAB_REPO, LAB_REF)], check=True)
import x3dlab

lab = x3dlab.up(tunnel=TUNNEL, show=SHOW_DASHBOARD, expose_mcp=EXPOSE_MCP, log_level=LOG_LEVEL, log_density=LOG_DENSITY,
                log_to_screen=LOG_TO_SCREEN, max_llm_runs=MAX_MODEL_RUNS, ngrok_domain=NGROK_DOMAIN or None)

**Open the dashboard** from the card. It shows the 3D view, the validator's verdicts and the live event stream, and it works on a phone when you use a tunnel.

**Chrome DevTools:** on the dashboard press <kbd>F12</kbd> (or <kbd>⌥⌘I</kbd>). The *Console* gets a styled line per event and a table of issues per scene; type `x3dlab.help()`. In *Performance*, record while a scene runs and look for the **x3d.lab** custom tracks (Validator, MCP, LLM, X_ITE). In *Network*, `events` is the live stream and every API call carries a *Server-Timing* header.

## 2 · API keys

Any one provider is enough. Paste a key, or store it once in Colab **Secrets** (the key icon in the left bar) as `OPENAI_API_KEY`, `ANTHROPIC_API_KEY` or `GEMINI_API_KEY` and allow this notebook to read it. **Test** lists the models your key can use and picks a sensible default; change it in the dropdown.

Keys stay inside this Colab kernel. They are removed from every log line and never sent to the dashboard, a tunnel or the x3d_mcp server.

In [ ]:
#@title 2 · API keys (OpenAI, Claude or Gemini)
x3dlab.keys.widget()

## 3 · Your first scene

- **simple** mode: the model writes the whole document. x3d_mcp checks it (schema and semantics), applies the fixes that have exactly one right answer (free, no model), and the model repairs whatever is left from the validator's report.
- **agent** mode: the model gets x3d_mcp tools (`describe_node`, `where_can_place`, `validate_x3d`, `validate_semantic`, `exact_fix_hints`, `repair_x3d`, …) and uses them *while* it writes, then submits.

`first draft valid` tells you whether the model got it right before any help. Without a key, `PROVIDER = "auto"` falls back to a built-in sample so you can still see the loop.

In [ ]:
#@title 3 · Make a scene
PROMPT = "A small lighthouse on a rocky island at dusk, with a slowly turning lamp and calm blue water."  #@param {type:"string"}
PROVIDER = "auto"  #@param ["auto", "anthropic", "openai", "gemini", "sample"]
MODE = "simple"  #@param ["simple", "agent"]
REPAIR_ROUNDS = 2  #@param {type:"slider", min:0, max:4, step:1}
X3D_VERSION = "4.0"  #@param ["4.0", "4.1"]

result = lab.scene(PROMPT, provider=PROVIDER, mode=MODE, rounds=REPAIR_ROUNDS, version=X3D_VERSION)
x3dlab.show_result(result, lab)

## 4 · Talk to the MCP server yourself

The same tools a model uses are plain function calls here. `lab.tool(name, **args)` calls one and returns its JSON.

In [ ]:
#@title 4 · Call x3d_mcp tools
tools = lab.client.run(lab.client.list_tools())
print(len(tools), "tools:", ", ".join(sorted(t.name for t in tools)), "\n")

node = lab.tool("describe_node", node_type="OrientationInterpolator")
print("OrientationInterpolator:", node["component"], "component · containerField", node["containerField"])
print("  fields:", ", ".join("%s (%s)" % (f["name"], f["type"]) for f in node["fields"]), "\n")

where = lab.tool("where_can_place", child="ImageTexture", limit=8)
print("Where can an ImageTexture go? %d places, for example:" % where["total"])
for row in where["rows"]:
    note = "  (needs containerField='%s')" % row["containerField"] if row["needsContainerField"] else ""
    print("  %s.%s%s" % (row["parent"], row["field"], note))
print()

verdict = lab.check(x3dlab.scenes.LANTERN)
print("The sample scene is", "valid" if verdict.ok else "not valid", "| warnings:", len(verdict.warnings))

## 5 · Connect Claude Code, Claude Desktop or Cursor to this Colab

With a tunnel on (cell 1, `TUNNEL = "cloudflared"`), the x3d_mcp server in this notebook is a remote MCP server. Copy one of the configs below into your client. Every tool call it makes shows up live in this notebook and on the dashboard.

In [ ]:
#@title 5 · MCP client configs
x3dlab.show_client_configs(lab)

## 6 · Break it!

A document meets four gates in order: the **XML parser** (is it XML at all?), the **XSD schema** for its X3D version (real node and field names, value patterns), x3d_mcp's **semantic check** (things a schema cannot express: DEF before USE, ROUTE types, interpolator lengths, profiles), and finally **nobody**: a scene can pass every check and still show nothing. Pick a defect, guess the gate, and check. On the dashboard, Break it also shows what the X_ITE browser itself says about the broken scene.

In [ ]:
#@title 6 · Break it
from x3dlab import breakit
breakit.game(lab)

## 7 · Copilot Validator Demo (private build)

Your private build of the browser-based X3D Copilot. The build includes the upstream author's icons and names, so it is **not** in the public repo: this cell loads your copy from Google Drive, a private GitHub release (needs `GITHUB_TOKEN` in Secrets), an upload, or a path. It is served next to the dashboard at `…/copilot/`.

In its DevTools console, `window.__x3dcopilot` exposes `viewer`, `editor`, `validator`, `runPipeline`, `tools` and `inspector`; for example `__x3dcopilot.validator.validate(__x3dcopilot.editor.getValue())`.

In [ ]:
#@title 7 · Load the Copilot demo { display-mode: "form" }
COPILOT_SOURCE = "drive"  #@param ["drive", "github", "upload", "path"]
COPILOT_PATH = "/content/drive/MyDrive/x3d.lab/Copilot_Validator_Demo.zip"  #@param {type:"string"}
COPILOT_REPO = ""  #@param {type:"string"}
COPILOT_TAG = "copilot-demo"  #@param {type:"string"}

copilot_url = lab.load_copilot(COPILOT_SOURCE, path=COPILOT_PATH or None, repo=COPILOT_REPO or None, tag=COPILOT_TAG)
x3dlab.ui.show("<a href='%s' target='_blank'>Open the Copilot Validator Demo</a>" % copilot_url)

## 8 · Status and stopping

`lab.status()` lists the processes, routes and recent jobs. The full event log is the JSONL file it names. `lab.down()` stops the server, the gateway and any tunnel; rerunning cell 1 also replaces everything cleanly.

| If… | Try |
|---|---|
| a tunnel self-test failed | rerun cell 1 (quick tunnels sometimes need a second try), or switch tunnel |
| the dashboard says *MCP down* | rerun cell 1 |
| the Colab link opens a blank page | use `SHOW_DASHBOARD = "iframe"`, or open the link in a new tab while signed in to Google |
| ngrok asks for a token | add `NGROK_AUTHTOKEN` in Secrets, or use `cloudflared` |
| a model call fails with 401 | the key is wrong or has no credit: cell 2, **Test** |

In [ ]:
#@title 8 · Status
import json
print(json.dumps(lab.status(), indent=1, default=str)[:3000])
# lab.down()